# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shenone77/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

A small logistic ranker compared to the frozen rule on the same clients and metric.

## Setup

Open this notebook from the repository or its Colab badge. The analysis uses FlyRank's bundled anonymized sample.

In [1]:
from pathlib import Path
import subprocess
import sys

if "google.colab" in sys.modules:
    root = Path("/content/flyrank-ml-internship")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Shenone77/flyrank-ml-internship.git", str(root)],
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.2", "scikit-learn>=1.4"], check=True)
else:
    start = Path.cwd().resolve()
    root = next(
        (path for path in (start, *start.parents) if (path / "work/notebooks/refresh_triage.py").is_file()),
        None,
    )
    if root is None:
        raise FileNotFoundError("Open the notebook inside a clone of the assignment repository.")

sys.path.insert(0, str(root / "work/notebooks"))
from refresh_triage import load_pages

pages = load_pages(root)
print(f"Loaded {len(pages):,} eligible pages from the FlyRank starter sample.")

Loaded 13,412 eligible pages from the FlyRank starter sample.


## 1. Method choice and why

Fit logistic regression with standardization learned only from training pages. Three pre-cutoff features give an editor a short explanation: log prior impressions, prior CTR, and age at the decision cutoff. This lets the weights reflect measured relationships with later material loss. A simple linear model is appropriate for a first comparison with a fixed rule.

FlyRank's [March 2026 research paper](../../docs/flyrank-seo-research-march-2026.pdf) leads with portfolio comparisons and treats its ML appendix as exploratory. This notebook similarly pairs a transparent comparator with an exploratory learned score.

## 2. Split design

Freeze the ML-07 split: one grouped 75%/25% split by client (seed 42). All pages from a client stay on one side. The share refers to clients; the numbers of pages per client differ. The feature and outcome windows are separated, and every page in this exercise comes from the same starter snapshot. This tests transfer across clients inside that snapshot.

In [2]:
import json
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from refresh_triage import (
    FEATURES, REVIEW_BUDGET, SEED, baseline_score, client_holdout, precision_at_k, ranked,
)

train, holdout = client_holdout(pages)
display(pd.DataFrame([
    {"Split": "Train", "Clients": train["client_id"].nunique(), "Pages": len(train),
     "Material-drop share": train["material_drop"].mean()},
    {"Split": "Holdout", "Clients": holdout["client_id"].nunique(), "Pages": len(holdout),
     "Material-drop share": holdout["material_drop"].mean()},
]).round(3))

,Split,Clients,Pages,Material-drop share
0,Train,21,12442,0.467
1,Holdout,7,970,0.853


## 3. Train and compare against the same baseline

Both methods score the same held-out rows and target. Precision@50 is primary, Precision@20 tests a tighter review budget, and average precision describes the entire ranking. The expected random-queue precision equals the held-out base rate. Exact ties use the same seeded order.

In [3]:
feature_names = list(FEATURES)
target = holdout["material_drop"]
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, random_state=SEED))
model.fit(train[feature_names], train["material_drop"])
probabilities = model.predict_proba(holdout[feature_names])[:, 1]
methods = {"Frozen exposure rule": baseline_score(holdout), "Logistic regression": probabilities}
metrics = {}
queues = {}
for method, scores in methods.items():
    ranked_test = ranked(holdout, scores)
    queues[method] = ranked_test
    metrics[method] = {
        "precision_at_20": precision_at_k(ranked_test, 20),
        "precision_at_50": precision_at_k(ranked_test),
        "average_precision": float(average_precision_score(target, scores)),
        "clients_in_top_50": int(ranked_test.head(REVIEW_BUDGET)["client_id"].nunique()),
    }

display(pd.DataFrame.from_dict(metrics, orient="index").round(3))
print(f"Held-out expected random precision: {target.mean():.1%}")

,precision_at_20,precision_at_50,average_precision,clients_in_top_50
Frozen exposure rule,0.85,0.84,0.881,2
Logistic regression,0.95,0.96,0.932,3


Held-out expected random precision: 85.3%


## 4. Errors and interpretation

Inspect the standardized coefficient signs and three real errors from the top-50 cutoff: two selected pages without a material drop and a material drop just outside the review budget. Display only rounded or binned fields. An error can be a near miss on either decision threshold, or a genuine reversal that needs data about demand or indexing to explain.

In [4]:
coefficients = model.named_steps["logisticregression"].coef_[0]
display(pd.Series(coefficients, index=feature_names, name="Standardized coefficient").round(3))

model_queue = queues["Logistic regression"].copy()
model_queue["review_rank"] = np.arange(1, len(model_queue) + 1)
top = model_queue.head(REVIEW_BUDGET)
beyond_budget = model_queue.iloc[REVIEW_BUDGET:]
false_alarms = top.loc[~top["material_drop"]].head(2)
missed = beyond_budget.loc[beyond_budget["material_drop"]].head(3 - len(false_alarms))
errors = pd.concat([
    false_alarms.assign(error="Selected without material drop"),
    missed.assign(error="Material drop outside budget"),
])
previous = errors["impressions_prev_30d"]
loss = previous - errors["impressions_last_30d"]
error_review = pd.DataFrame({
    "Rank": errors["review_rank"],
    "Error": errors["error"],
    "Prior impressions (nearest 100)": previous.div(100).round().mul(100).astype(int),
    "Prior CTR (%)": errors["prior_ctr_pct"].round(2),
    "Age at decision (30-day band)": errors["age_at_cutoff_days"].floordiv(30).mul(30),
    "Observed loss (%)": (100 * loss / previous).round(1),
    "Lost at least 150": loss.ge(150),
})
display(error_review.reset_index(drop=True))
assert len(error_review) == 3

log_prior_impressions   -0.133
prior_ctr_pct           -0.511
age_at_cutoff_days      -0.441
Name: Standardized coefficient, dtype: float64

,Rank,Error,Prior impressions (nearest 100),Prior CTR (%),Age at decision (30-day band),Observed loss (%),Lost at least 150
0,2,Selected without material drop,300,0.0,60,41.4,False
1,40,Selected without material drop,500,0.0,60,15.5,False
2,51,Material drop outside budget,500,0.0,90,73.8,True


**Measured result.** The rule found 42 material drops in its top 50. The logistic score found 48. The held-out base rate already exceeded 85%, so the absolute gain should be interpreted against a high floor. Prior CTR and age had negative fitted coefficients. They are associations within these data. The top 50 logistic picks came from only three clients, and the training and held-out base rates diverged substantially. A later time window and a per-client review budget would test whether this advantage survives under a more representative editorial workload.

## 5. Self-check and metric receipt

Verify the baseline matches ML-07 on the same population and that the model uses only the three pre-cutoff predictors. The small JSON contains aggregated metrics and versions so the comparison can be traced to this run.

In [5]:
assert set(feature_names) == {"log_prior_impressions", "prior_ctr_pct", "age_at_cutoff_days"}
assert set(train["client_id"]).isdisjoint(holdout["client_id"])
assert len(probabilities) == len(target)
assert np.isfinite(probabilities).all()
receipt = {
    "seed": SEED,
    "review_budget": REVIEW_BUDGET,
    "training_pages": len(train),
    "training_clients": int(train["client_id"].nunique()),
    "holdout_pages": len(holdout),
    "holdout_clients": int(holdout["client_id"].nunique()),
    "holdout_base_rate": float(target.mean()),
    "features": feature_names,
    "metrics": metrics,
    "versions": {"pandas": pd.__version__, "scikit_learn": sklearn.__version__},
}
path = root / "work/outputs/refresh_triage_metrics.json"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
print("Self-check passed. Wrote work/outputs/refresh_triage_metrics.json.")

Self-check passed. Wrote work/outputs/refresh_triage_metrics.json.
